# Main Pilot — Temporal DFE in Pythia 410M

**Design** (approved 2026-04-18):
- 8 checkpoints: 512, 1000, 2000, 4000, 8000, 16000, 64000, 143000 (dense in early phase)
- **144 fixed heads**: `{0, 3, 6, 9, 12, 15}` × 24 layers — SAME heads across all checkpoints for identity tracking
- 24 layer ablations (exhaustive) per checkpoint
- 30 Type A parametric perturbations (α=1.0) per checkpoint, CLT-limit reference
- Total: **1,584 ablations**
- **Float32 eval** (fp16 floor destroyed signal at step 1000 in minimal pilot)

**Primary metrics** (threshold-free):
1. Gini coefficient of |delta| distribution (inequality of head importance)
2. Effective N via entropy (2^H) — biological analog of effective number of species

**Secondary metrics**:
3. Differentiation count (#heads with |delta|>5e-4) — for intuitive tables
4. Distribution fits (Student's t vs Normal, ΔAIC)
5. Gamma β on deleterious tail (bootstrap CI)

**Robustness safeguards:**
- Baseline recomputed per checkpoint on identical validation batches
- Raw deltas saved to CSV, one row per ablation, append-mode (resilient to Colab disconnect)
- Bitwise save/restore hash check on every ablation
- Resume logic: skip already-completed checkpoints

**Expected runtime on A100 40GB float32**: ~2.5-3.5 hours for full run.

## 1. Dependencies

In [ ]:
!pip install -q transformers datasets torch accelerate scipy

## 2. Mount Drive & config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json, csv, hashlib, gc

DRIVE_ROOT = '/content/drive/MyDrive/DFE_research'
DATA_DIR = os.path.join(DRIVE_ROOT, 'data/main_pilot')
os.makedirs(DATA_DIR, exist_ok=True)

CSV_PATH = os.path.join(DATA_DIR, 'all_ablations.csv')
RESULTS_JSON = os.path.join(DATA_DIR, 'main_pilot_results.json')
PROGRESS_JSON = os.path.join(DATA_DIR, 'progress.json')

# ── Pilot configuration ────────────────────────────────────────────────────
MODEL_NAME = 'EleutherAI/pythia-410m-deduped'
CHECKPOINTS = [512, 1000, 2000, 4000, 8000, 16000, 64000, 143000]
FIXED_HEADS = [0, 3, 6, 9, 12, 15]          # same heads sampled at every checkpoint
N_LAYERS = 24
N_HEADS = 16
INTERMEDIATE_SIZE = 4096

TYPE_A_ALPHA = 1.0                           # full-std parametric noise (CLT-limit reference)
N_TYPE_A = 30                                # Type A perturbations per checkpoint

EVAL_N_BATCHES = 25
EVAL_BATCH_SIZE = 4                          # float32 on A100: halve from fp16
EVAL_SEQ_LEN = 2048
SEED = 42
NEUTRAL_THRESHOLD = 5e-4                     # for secondary 'differentiation count' metric

print(f'Data dir: {DATA_DIR}')
print(f'Total ablations planned: {len(CHECKPOINTS)} × ({len(FIXED_HEADS)}×{N_LAYERS} + {N_LAYERS} + {N_TYPE_A}) = '
      f'{len(CHECKPOINTS) * (len(FIXED_HEADS)*N_LAYERS + N_LAYERS + N_TYPE_A)}')

## 3. Imports and helper functions

In [ ]:
import torch
import numpy as np
from scipy import stats as sp_stats
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True  # TF32 on A100 = near-float32 precision, near-fp16 speed
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('TF32 enabled (float32 semantics with A100 tensor cores)')


def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)


def tensor_hash(t):
    """Bitwise hash for verifying save/restore correctness."""
    return hashlib.sha256(t.detach().cpu().contiguous().numpy().tobytes()).hexdigest()[:16]


def tensor_checksum(t):
    """Cheap drift check — just sum of abs values."""
    return float(t.detach().abs().sum().item())


# ── CSV row writer (append-mode, resilient) ────────────────────────────────
CSV_FIELDS = ['checkpoint', 'perturbation_type', 'subtype', 'layer_idx', 'head_idx',
              'seed', 'baseline_loss', 'perturbed_loss', 'delta', 'elapsed_sec']

def append_row(row):
    write_header = not os.path.exists(CSV_PATH)
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if write_header:
            w.writeheader()
        w.writerow(row)


# ── Gini and effective N (threshold-free differentiation metrics) ──────────
def gini(values):
    """Gini coefficient of |values|. 0 = uniform, 1 = max concentration."""
    x = np.abs(np.array(values))
    if x.sum() == 0:
        return 0.0
    x_sorted = np.sort(x)
    n = len(x)
    cumsum = np.cumsum(x_sorted)
    return float((n + 1 - 2 * np.sum(cumsum) / cumsum[-1]) / n)


def effective_n(values):
    """Shannon-entropy-based effective number of important heads."""
    x = np.abs(np.array(values))
    total = x.sum()
    if total == 0:
        return float(len(x))
    p = x / total
    p_nz = p[p > 0]
    H = -np.sum(p_nz * np.log2(p_nz))
    return float(2 ** H)


# ── Bootstrap for any stat ─────────────────────────────────────────────────
def bootstrap_stat(values, stat_fn, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    x = np.array(values)
    bs = np.array([stat_fn(rng.choice(x, size=len(x), replace=True)) for _ in range(n_boot)])
    return float(np.median(bs)), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


# ── Full DFE analysis ──────────────────────────────────────────────────────
def analyze_dfe(deltas):
    d = np.array(deltas)
    n = len(d)
    mu_n, sig_n = sp_stats.norm.fit(d)
    aic_n = 2*2 - 2*np.sum(sp_stats.norm.logpdf(d, mu_n, sig_n))
    df_t, loc_t, sc_t = sp_stats.t.fit(d)
    aic_t = 2*3 - 2*np.sum(sp_stats.t.logpdf(d, df_t, loc_t, sc_t))
    loc_l, sc_l = sp_stats.laplace.fit(d)
    aic_l = 2*2 - 2*np.sum(sp_stats.laplace.logpdf(d, loc_l, sc_l))

    # gamma β on deleterious tail, fp16-noise-filtered
    neg = -d[d < -1e-4]
    if len(neg) >= 5:
        try:
            beta_med, beta_lo, beta_hi = bootstrap_stat(
                neg, lambda x: sp_stats.gamma.fit(x, floc=0)[0])
        except Exception:
            beta_med, beta_lo, beta_hi = float('nan'), float('nan'), float('nan')
    else:
        beta_med, beta_lo, beta_hi = float('nan'), float('nan'), float('nan')

    return {
        'n': n, 'mean': float(np.mean(d)), 'std': float(np.std(d)),
        'skewness': float(sp_stats.skew(d)), 'excess_kurtosis': float(sp_stats.kurtosis(d)),
        'f_positive': float(np.mean(d > 0)),
        'aic_normal': float(aic_n), 'aic_student_t': float(aic_t), 'aic_laplace': float(aic_l),
        'delta_aic_t_vs_normal': float(aic_n - aic_t),
        'delta_aic_laplace_vs_normal': float(aic_n - aic_l),
        'student_t_df': float(df_t),
        'gamma_beta_median': beta_med, 'gamma_beta_lo95': beta_lo, 'gamma_beta_hi95': beta_hi,
        'n_negative_above_noise': int(len(neg)),
        # Threshold-free primary metrics
        'gini': gini(d), 'effective_n': effective_n(d),
        # Secondary threshold-dependent
        'n_differentiated_5e4': int(np.sum(np.abs(d) > 5e-4)),
        'n_differentiated_1e3': int(np.sum(np.abs(d) > 1e-3)),
    }

## 4. Validation data (wikitext-103 train stream)

In [ ]:
log('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

log('Streaming wikitext-103 train split for validation tokens...')
ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='train', streaming=True)

total_needed = EVAL_N_BATCHES * EVAL_BATCH_SIZE * EVAL_SEQ_LEN
all_tokens = []
for example in ds:
    text = example.get('text', '')
    if len(text.strip()) < 50:
        continue
    tokens = tokenizer(text, return_tensors='pt', truncation=False)['input_ids'].squeeze()
    if tokens.dim() == 0:
        continue
    all_tokens.append(tokens)
    if sum(t.numel() for t in all_tokens) >= total_needed * 1.2:
        break

all_tokens = torch.cat(all_tokens)[:total_needed]
token_batches = all_tokens.reshape(EVAL_N_BATCHES, EVAL_BATCH_SIZE, EVAL_SEQ_LEN)
log(f'Validation tokens: {token_batches.shape} = {token_batches.numel():,} total')
# Cache on drive so exact same tokens used across sessions
torch.save(token_batches, os.path.join(DATA_DIR, 'validation_tokens.pt'))

## 5. Model-level operations (evaluation + perturbations)

In [ ]:
@torch.no_grad()
def evaluate_loss(model, batches):
    total = 0.0
    for i in range(batches.shape[0]):
        ids = batches[i].to(device)
        total += model(input_ids=ids, labels=ids).loss.item()
    return total / batches.shape[0]


def load_model_fp32(step):
    log(f'Loading {MODEL_NAME} at step{step} in float32 (TF32 matmul)...')
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, revision=f'step{step}', torch_dtype=torch.float32
    ).to(device)
    m.eval()
    return m


# ── Head ablation (output-projection zeroing) ──────────────────────────────
def ablate_head(model, layer_idx, head_idx):
    head_dim = model.config.hidden_size // model.config.num_attention_heads
    s, e = head_idx * head_dim, (head_idx + 1) * head_dim
    w = model.gpt_neox.layers[layer_idx].attention.dense.weight
    saved = w.data.clone()
    w.data[:, s:e] = 0
    return ('head', layer_idx, saved)


def ablate_layer(model, layer_idx):
    block = model.gpt_neox.layers[layer_idx]
    saved = {name: p.data.clone() for name, p in block.named_parameters()}
    for _, p in block.named_parameters():
        p.data.zero_()
    return ('layer', layer_idx, saved)


def perturb_type_a(model, block_idx, alpha, seed):
    """Parametric noise at fixed seed for reproducibility."""
    g = torch.Generator(device=device).manual_seed(seed)
    block = model.gpt_neox.layers[block_idx]
    saved = {name: p.data.clone() for name, p in block.named_parameters()}
    for _, p in block.named_parameters():
        noise = torch.randn(p.shape, generator=g, device=device, dtype=p.dtype) * (p.data.std() * alpha)
        p.data.add_(noise)
    return ('type_a', block_idx, saved)


def restore(model, handle):
    kind, idx, saved = handle
    if kind == 'head':
        model.gpt_neox.layers[idx].attention.dense.weight.data.copy_(saved)
    elif kind in ('layer', 'type_a'):
        block = model.gpt_neox.layers[idx]
        for name, p in block.named_parameters():
            p.data.copy_(saved[name])


# ── Verify save/restore bitwise on a random op ─────────────────────────────
def verify_bitwise(model, op_fn, op_args, check_layer=5):
    w = model.gpt_neox.layers[check_layer].attention.dense.weight
    h_before = tensor_hash(w.data)
    handle = op_fn(model, **op_args)
    h_abl = tensor_hash(w.data)
    restore(model, handle)
    h_after = tensor_hash(w.data)
    return h_before, h_abl, h_after

## 6. Sanity check: bitwise save/restore on fp32 model

In [ ]:
log('Sanity-loading smallest checkpoint for bitwise verification...')
_m = load_model_fp32(CHECKPOINTS[0])

# Test 1: head ablation
h1, h2, h3 = verify_bitwise(_m, ablate_head, {'layer_idx': 5, 'head_idx': 7})
log(f'head ablation: before={h1}  abl={h2}  after={h3}  OK={h1==h3 and h1!=h2}')
assert h1 == h3 and h1 != h2, 'head ablation save/restore broken'

# Test 2: layer ablation (checks different tensor)
h1, h2, h3 = verify_bitwise(_m, ablate_layer, {'layer_idx': 5}, check_layer=5)
log(f'layer ablation: before={h1}  abl={h2}  after={h3}  OK={h1==h3 and h1!=h2}')
assert h1 == h3 and h1 != h2, 'layer ablation save/restore broken'

# Test 3: type A noise
h1, h2, h3 = verify_bitwise(_m, perturb_type_a, {'block_idx': 5, 'alpha': 1.0, 'seed': 0}, check_layer=5)
log(f'type A noise: before={h1}  abl={h2}  after={h3}  OK={h1==h3 and h1!=h2}')
assert h1 == h3 and h1 != h2, 'type_a save/restore broken'

del _m; gc.collect(); torch.cuda.empty_cache()
log('All three perturbation types verified bitwise.')

## 7. Main loop — per checkpoint

Resume logic: any (checkpoint, perturbation_type, layer, head) tuple already in the CSV is skipped.
Cheap checksum drift check after every 50 ablations.

In [ ]:
# Load existing CSV to find completed items
completed = set()
if os.path.exists(CSV_PATH):
    import pandas as pd
    df = pd.read_csv(CSV_PATH)
    completed = set(df.apply(lambda r: (int(r['checkpoint']), str(r['perturbation_type']),
                                         int(r['layer_idx']), int(r['head_idx'])), axis=1))
    log(f'CSV exists with {len(completed)} completed ablations. Will skip those.')


def run_checkpoint(step):
    t_cp = time.time()
    log(f'\n{"=" * 70}\nCHECKPOINT step {step}\n{"=" * 70}')

    # Determine work
    todo_heads = [(layer, head) for layer in range(N_LAYERS) for head in FIXED_HEADS
                  if (step, 'head', layer, head) not in completed]
    todo_layers = [layer for layer in range(N_LAYERS)
                   if (step, 'layer', layer, -1) not in completed]
    todo_type_a = [(i,) for i in range(N_TYPE_A)
                   if (step, 'type_a', i, -1) not in completed]
    total_todo = len(todo_heads) + len(todo_layers) + len(todo_type_a)

    if total_todo == 0:
        log(f'step {step}: all ablations already done, skipping.')
        return
    log(f'step {step}: todo = {len(todo_heads)} heads + {len(todo_layers)} layers + '
        f'{len(todo_type_a)} Type A = {total_todo}')

    model = load_model_fp32(step)

    log('Computing baseline loss...')
    baseline = evaluate_loss(model, token_batches)
    log(f'  baseline_loss = {baseline:.6f}  |  ppl = {np.exp(baseline):.2f}')

    # Reference checksum for drift monitoring (pick a fixed tensor that is restored every time)
    drift_ref_tensor = model.gpt_neox.layers[5].attention.dense.weight
    drift_ref_checksum = tensor_checksum(drift_ref_tensor)
    count = 0

    # ── Heads ─────────────────────────────────────────────────────────────
    rng_seeds = np.random.default_rng(SEED)  # not needed for head, but for type_a reproducibility
    for (layer, head) in todo_heads:
        t0 = time.time()
        handle = ablate_head(model, layer, head)
        loss = evaluate_loss(model, token_batches)
        delta = -(loss - baseline) / abs(baseline)
        restore(model, handle)
        append_row({
            'checkpoint': step, 'perturbation_type': 'head', 'subtype': 'output_zeroing',
            'layer_idx': layer, 'head_idx': head, 'seed': -1,
            'baseline_loss': baseline, 'perturbed_loss': loss, 'delta': delta,
            'elapsed_sec': time.time() - t0,
        })
        count += 1
        if count % 30 == 0:
            drift = abs(tensor_checksum(drift_ref_tensor) - drift_ref_checksum)
            log(f'  [{count}/{total_todo}] L{layer:>2}H{head:>2} delta={delta:+.5f}  drift={drift:.2e}')
            assert drift < 1e-3, f'Checksum drift exceeded tolerance: {drift}'

    # ── Layer ablations (exhaustive) ──────────────────────────────────────
    for layer in todo_layers:
        t0 = time.time()
        handle = ablate_layer(model, layer)
        loss = evaluate_loss(model, token_batches)
        delta = -(loss - baseline) / abs(baseline)
        restore(model, handle)
        append_row({
            'checkpoint': step, 'perturbation_type': 'layer', 'subtype': 'full_zeroing',
            'layer_idx': layer, 'head_idx': -1, 'seed': -1,
            'baseline_loss': baseline, 'perturbed_loss': loss, 'delta': delta,
            'elapsed_sec': time.time() - t0,
        })
        count += 1
        log(f'  [{count}/{total_todo}] LAYER L{layer:>2} delta={delta:+.5f}')

    # ── Type A parametric noise ───────────────────────────────────────────
    for (i,) in todo_type_a:
        t0 = time.time()
        # Deterministic: block = i % N_LAYERS, seed = SEED*1000 + i
        block_idx = int(i) % N_LAYERS
        p_seed = SEED * 1000 + int(i)
        handle = perturb_type_a(model, block_idx, TYPE_A_ALPHA, p_seed)
        loss = evaluate_loss(model, token_batches)
        delta = -(loss - baseline) / abs(baseline)
        restore(model, handle)
        append_row({
            'checkpoint': step, 'perturbation_type': 'type_a', 'subtype': f'noise_alpha{TYPE_A_ALPHA}',
            'layer_idx': block_idx, 'head_idx': -1, 'seed': p_seed,
            'baseline_loss': baseline, 'perturbed_loss': loss, 'delta': delta,
            'elapsed_sec': time.time() - t0,
        })
        count += 1
        if count % 10 == 0 or count == total_todo:
            log(f'  [{count}/{total_todo}] TYPE_A i={i} block={block_idx} delta={delta:+.5f}')

    elapsed_min = (time.time() - t_cp) / 60
    log(f'step {step} done in {elapsed_min:.1f} min ({total_todo} ablations)')

    # Save progress marker
    progress = {}
    if os.path.exists(PROGRESS_JSON):
        with open(PROGRESS_JSON) as f: progress = json.load(f)
    progress[str(step)] = {'completed': True, 'elapsed_min': elapsed_min, 'baseline_loss': baseline}
    with open(PROGRESS_JSON, 'w') as f:
        json.dump(progress, f, indent=2)

    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Run the full sweep
t_all = time.time()
for step in CHECKPOINTS:
    run_checkpoint(step)

log(f'\n\n>>> ALL CHECKPOINTS COMPLETE in {(time.time()-t_all)/60:.1f} min <<<')

## 8. Analysis: per-checkpoint DFE metrics (heads only for core result)

In [ ]:
import pandas as pd
df = pd.read_csv(CSV_PATH)
log(f'Loaded {len(df)} rows from CSV')
df_heads = df[df['perturbation_type'] == 'head'].copy()
df_layers = df[df['perturbation_type'] == 'layer'].copy()
df_typea = df[df['perturbation_type'] == 'type_a'].copy()

results = {}
for step in CHECKPOINTS:
    sub = df_heads[df_heads['checkpoint'] == step]
    if len(sub) < 10:
        continue
    deltas = sub['delta'].values
    results[step] = analyze_dfe(deltas)
    results[step]['baseline_loss'] = float(sub['baseline_loss'].iloc[0])

# Summary table
print(f'{"step":>7} | {"gini":>6} {"eff_N":>6} | {"skew":>7} {"kurt":>7} | '
      f'{"dAIC(t-n)":>9} | {"β med":>6} {"β_95CI":>16} | {"n_diff(5e-4)":>12}')
print('-' * 95)
for step in CHECKPOINTS:
    if step not in results:
        continue
    r = results[step]
    beta_ci = f'[{r["gamma_beta_lo95"]:.2f},{r["gamma_beta_hi95"]:.2f}]' if not np.isnan(r['gamma_beta_median']) else 'NaN'
    beta_med = f'{r["gamma_beta_median"]:.2f}' if not np.isnan(r['gamma_beta_median']) else 'NaN'
    print(f'{step:>7} | {r["gini"]:>6.3f} {r["effective_n"]:>6.1f} | '
          f'{r["skewness"]:>+7.3f} {r["excess_kurtosis"]:>+7.3f} | '
          f'{r["delta_aic_t_vs_normal"]:>+9.2f} | {beta_med:>6} {beta_ci:>16} | {r["n_differentiated_5e4"]:>12}')

with open(RESULTS_JSON, 'w') as f:
    json.dump(results, f, indent=2, default=str)

## 9. Figure 1 — Individual head trajectories (PRIMARY)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIG_DIR = os.path.join(DRIVE_ROOT, 'figures/main_pilot')
os.makedirs(FIG_DIR, exist_ok=True)

# Build (layer, head) → trajectory dict
traj = {}
for (L, H), g in df_heads.groupby(['layer_idx', 'head_idx']):
    g = g.sort_values('checkpoint')
    if len(g) == len(CHECKPOINTS):
        traj[(L, H)] = g.set_index('checkpoint')['delta'].to_dict()

# Classify each head
def classify(t):
    init = abs(t[CHECKPOINTS[0]])
    final = abs(t[CHECKPOINTS[-1]])
    if final < 5e-4:
        return 'never-critical'
    if init > 5e-4 and final > 5e-4:
        return 'born-critical'
    if init < 1e-4 and final > 5e-4:
        return 'emergent'
    return 'growing'

class_color = {'born-critical': 'purple', 'emergent': 'crimson',
               'growing': 'steelblue', 'never-critical': 'lightgray'}
class_counts = {k: 0 for k in class_color}

fig, ax = plt.subplots(figsize=(11, 6.5))
for (L, H), t in traj.items():
    cls = classify(t)
    class_counts[cls] += 1
    alpha = 0.8 if cls != 'never-critical' else 0.15
    lw = 1.6 if cls != 'never-critical' else 0.6
    ys = [abs(t[s]) for s in CHECKPOINTS]
    ax.plot(CHECKPOINTS, ys, color=class_color[cls], alpha=alpha, linewidth=lw, marker='o', markersize=3)

for cls, col in class_color.items():
    ax.plot([], [], color=col, marker='o', linewidth=1.5, label=f'{cls} (n={class_counts[cls]})')
ax.axhline(5e-4, color='black', linestyle='--', alpha=0.4, label='threshold 5e-4')
ax.axhline(1e-4, color='black', linestyle=':', alpha=0.3, label='fp16-like noise floor 1e-4')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Training step', fontsize=12)
ax.set_ylabel('|delta| — magnitude of ablation effect', fontsize=12)
ax.set_title('Functional differentiation of attention heads during training\n(each line = one head tracked across 8 checkpoints)', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig1_head_trajectories.png', dpi=220, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig1_head_trajectories.pdf', bbox_inches='tight')
plt.show()
print(f'Classification counts: {class_counts}')

## 10. Figure 2 — Differentiation metrics (threshold-free)

In [ ]:
# Bootstrap Gini and Effective N per checkpoint
gini_tr, effN_tr = [], []
for step in CHECKPOINTS:
    sub = df_heads[df_heads['checkpoint'] == step]['delta'].values
    if len(sub) < 10:
        gini_tr.append((None, None, None)); effN_tr.append((None, None, None)); continue
    gini_tr.append(bootstrap_stat(sub, gini))
    effN_tr.append(bootstrap_stat(sub, effective_n))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

med_g = [x[0] for x in gini_tr]; lo_g = [x[1] for x in gini_tr]; hi_g = [x[2] for x in gini_tr]
ax1.errorbar(CHECKPOINTS, med_g, yerr=[np.subtract(med_g, lo_g), np.subtract(hi_g, med_g)],
             fmt='o-', markersize=9, linewidth=2, capsize=5, color='darkgreen')
ax1.set_xscale('log')
ax1.set_xlabel('Training step', fontsize=11)
ax1.set_ylabel('Gini coefficient', fontsize=11)
ax1.set_title('Inequality of head importance\n(0 = uniform, 1 = full concentration)', fontsize=12)
ax1.grid(True, alpha=0.3)

med_e = [x[0] for x in effN_tr]; lo_e = [x[1] for x in effN_tr]; hi_e = [x[2] for x in effN_tr]
ax2.errorbar(CHECKPOINTS, med_e, yerr=[np.subtract(med_e, lo_e), np.subtract(hi_e, med_e)],
             fmt='s-', markersize=9, linewidth=2, capsize=5, color='darkred')
ax2.axhline(144, color='gray', linestyle='--', alpha=0.5, label='N total heads (uniform limit)')
ax2.set_xscale('log')
ax2.set_xlabel('Training step', fontsize=11)
ax2.set_ylabel('Effective N (2^H)', fontsize=11)
ax2.set_title('Effective number of important heads\n(entropy-based, biological analog of species diversity)', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.suptitle('Threshold-free differentiation metrics (bootstrap 95% CI)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig2_differentiation_metrics.png', dpi=220, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig2_differentiation_metrics.pdf', bbox_inches='tight')
plt.show()

## 11. Figure 3 — Emergence of DFE shape (selected checkpoints)

In [ ]:
sel = [CHECKPOINTS[0], CHECKPOINTS[2], CHECKPOINTS[5], CHECKPOINTS[-1]]  # 4 checkpoints
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for i, step in enumerate(sel):
    ax = axes[i]
    d = df_heads[df_heads['checkpoint'] == step]['delta'].values
    ax.hist(d, bins=40, density=True, alpha=0.65, color=plt.cm.viridis(i/3), edgecolor='none')
    df_t, lt, st = sp_stats.t.fit(d); mu, si = sp_stats.norm.fit(d)
    x = np.linspace(np.percentile(d, 0.3), np.percentile(d, 99.7), 300)
    ax.plot(x, sp_stats.t.pdf(x, df_t, lt, st), 'k-', lw=2, label=f"t (df={df_t:.1f})")
    ax.plot(x, sp_stats.norm.pdf(x, mu, si), 'r--', lw=1.3, alpha=0.8, label='Normal')
    ax.axvline(0, color='gray', linestyle=':', alpha=0.4)
    ax.axvline(-5e-4, color='red', linestyle=':', alpha=0.3)
    r = results[step]
    ax.set_title(f'step {step:,}\nGini={r["gini"]:.2f}, eff_N={r["effective_n"]:.0f}/{len(d)}', fontsize=11)
    ax.set_xlabel('delta')
    if i == 0: ax.set_ylabel('density')
    ax.legend(fontsize=8)
fig.suptitle('Emergence of DFE shape: from delta-peak-plus-catastrophes to continuous Student\'s t', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig3_dfe_emergence.png', dpi=220, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig3_dfe_emergence.pdf', bbox_inches='tight')
plt.show()

## 12. Figure 4 — Quantitative distribution fit

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: ΔAIC bars
daic_t = [results[s]['delta_aic_t_vs_normal'] for s in CHECKPOINTS if s in results]
daic_l = [results[s]['delta_aic_laplace_vs_normal'] for s in CHECKPOINTS if s in results]
steps_v = [s for s in CHECKPOINTS if s in results]
x = np.arange(len(steps_v)); w = 0.4
ax1.bar(x-w/2, daic_t, w, label='Student\'s t', color='steelblue')
ax1.bar(x+w/2, daic_l, w, label='Laplace', color='coral')
ax1.axhline(2, color='k', linestyle='--', alpha=0.4, label='ΔAIC=2')
ax1.axhline(10, color='g', linestyle='--', alpha=0.3, label='ΔAIC=10 (strong)')
ax1.set_yscale('symlog')
ax1.set_xticks(x); ax1.set_xticklabels([f'{s:,}' for s in steps_v], rotation=45)
ax1.set_xlabel('Training step'); ax1.set_ylabel('ΔAIC vs Normal')
ax1.set_title('Distribution fit superiority over Normal')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3, axis='y')

# Right: β trajectory with bootstrap CI
beta_med = [results[s]['gamma_beta_median'] for s in steps_v]
beta_lo = [results[s]['gamma_beta_lo95'] for s in steps_v]
beta_hi = [results[s]['gamma_beta_hi95'] for s in steps_v]
valid = [i for i, b in enumerate(beta_med) if not np.isnan(b)]
if valid:
    xs = [steps_v[i] for i in valid]
    meds = [beta_med[i] for i in valid]
    los = [meds[j] - beta_lo[valid[j]] for j in range(len(valid))]
    his = [beta_hi[valid[j]] - meds[j] for j in range(len(valid))]
    ax2.errorbar(xs, meds, yerr=[los, his], fmt='o-', markersize=9, lw=2, capsize=6, color='navy')
ax2.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='exponential (β=1)')
ax2.axhline(0.3, color='red', linestyle=':', alpha=0.5, label='VSV β≈0.3')
ax2.axhline(0.7, color='green', linestyle=':', alpha=0.5, label='yeast β≈0.7')
ax2.set_xscale('log')
ax2.set_xlabel('Training step'); ax2.set_ylabel('Gamma shape β (|delta|>1e-4 filtered)')
ax2.set_title('β trajectory with bootstrap 95% CI')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_distribution_fits.png', dpi=220, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig4_distribution_fits.pdf', bbox_inches='tight')
plt.show()

## 13. Figure 5 — Layer-level structure

In [ ]:
# Heatmap: layer × checkpoint, value = |delta|
mat = np.full((N_LAYERS, len(CHECKPOINTS)), np.nan)
for i, step in enumerate(CHECKPOINTS):
    sub = df_layers[df_layers['checkpoint'] == step]
    for _, row in sub.iterrows():
        mat[int(row['layer_idx']), i] = abs(row['delta'])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(mat, aspect='auto', cmap='viridis', norm=matplotlib.colors.LogNorm(vmin=1e-4, vmax=mat[~np.isnan(mat)].max() or 1))
ax.set_xticks(range(len(CHECKPOINTS))); ax.set_xticklabels([f'{s:,}' for s in CHECKPOINTS], rotation=45)
ax.set_yticks(range(N_LAYERS)); ax.set_yticklabels([f'L{i}' for i in range(N_LAYERS)])
ax.set_xlabel('Training step'); ax.set_ylabel('Layer')
ax.set_title('Layer ablation magnitude across training (log color scale)')
plt.colorbar(im, ax=ax, label='|delta|')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig5_layer_heatmap.png', dpi=220, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig5_layer_heatmap.pdf', bbox_inches='tight')
plt.show()

## 14. Figure 6 — Hierarchy of perturbations at final checkpoint

In [ ]:
final = CHECKPOINTS[-1]
d_typea = df_typea[df_typea['checkpoint'] == final]['delta'].values
d_head = df_heads[df_heads['checkpoint'] == final]['delta'].values
d_layer = df_layers[df_layers['checkpoint'] == final]['delta'].values

fig, ax = plt.subplots(figsize=(10, 5.5))
bins = np.linspace(min(d_typea.min(), d_head.min(), d_layer.min()) * 1.05,
                   max(d_typea.max(), d_head.max(), d_layer.max()) * 1.05, 60)
ax.hist(d_typea, bins=bins, density=True, alpha=0.55, label=f'Type A (noise α=1.0)  n={len(d_typea)}', color='gray')
ax.hist(d_head, bins=bins, density=True, alpha=0.55, label=f'Head ablation  n={len(d_head)}', color='steelblue')
ax.hist(d_layer, bins=bins, density=True, alpha=0.55, label=f'Layer ablation  n={len(d_layer)}', color='darkred')
ax.axvline(0, color='k', linestyle=':', alpha=0.4)
ax.set_xlabel('delta'); ax.set_ylabel('density')
ax.set_title(f'Perturbation granularity hierarchy at step {final:,}\n'
             f'(distributed noise → head → layer: increasing discreteness, heavier tails)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig6_hierarchy.png', dpi=220, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig6_hierarchy.pdf', bbox_inches='tight')
plt.show()

# Quick comparison of metrics
print(f'\nAt step {final}:')
for name, d in [('Type A', d_typea), ('Head', d_head), ('Layer', d_layer)]:
    df_t, lt, st = sp_stats.t.fit(d); mu, si = sp_stats.norm.fit(d)
    aic_t = 2*3 - 2*np.sum(sp_stats.t.logpdf(d, df_t, lt, st))
    aic_n = 2*2 - 2*np.sum(sp_stats.norm.logpdf(d, mu, si))
    print(f'  {name:>7}  n={len(d):>3}  std={d.std():.5f}  skew={sp_stats.skew(d):+.3f}  '
          f'kurt={sp_stats.kurtosis(d):+.3f}  ΔAIC(t-n)={aic_n-aic_t:+.2f}  t_df={df_t:.2f}')

## 15. Final summary

In [ ]:
print('=' * 80)
print('MAIN PILOT COMPLETE — PRIMARY RESULTS')
print('=' * 80)

print(f'\n{"step":>7} | {"baseline":>9} | {"gini":>6} {"eff_N":>6} | {"n_diff(5e-4)":>12}')
print('-' * 55)
for step in CHECKPOINTS:
    if step not in results:
        continue
    r = results[step]
    print(f'{step:>7} | {r["baseline_loss"]:>9.4f} | {r["gini"]:>6.3f} {r["effective_n"]:>6.1f} | '
          f'{r["n_differentiated_5e4"]:>12}')

# Trend tests
g = [results[s]['gini'] for s in CHECKPOINTS if s in results]
eN = [results[s]['effective_n'] for s in CHECKPOINTS if s in results]
nd = [results[s]['n_differentiated_5e4'] for s in CHECKPOINTS if s in results]

from scipy.stats import spearmanr
idx = list(range(len(g)))
print(f'\nTrend tests (Spearman rank correlation with training step):')
print(f'  Gini:           ρ = {spearmanr(idx, g).correlation:+.3f}  (expected +)')
print(f'  Effective N:    ρ = {spearmanr(idx, eN).correlation:+.3f}  (expected −, since eff_N inverse of concentration)')
print(f'  n_diff (5e-4):  ρ = {spearmanr(idx, nd).correlation:+.3f}  (expected +)')

print(f'\nClassification of 144 tracked heads:')
for cls, ct in class_counts.items():
    print(f'  {cls:>15}: {ct:>3}  ({100*ct/144:.0f}%)')

print(f'\nAll data: {CSV_PATH}')
print(f'Figures:  {FIG_DIR}/')
print(f'Results:  {RESULTS_JSON}')